In [1]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/249de2a1bc6c07516904de4a27993d56eaa9e2ad/01_data/processed/df_cleaned_ver2.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [2]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42602, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      0
negative              0
positive              0
neutral               0
compound              0
cyberbullying_type    0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [3]:
# ============================================================
# CELL 1: DATA PREPARATION
# WORD TF-IDF + SENTIMENT FEATURES
# DATASET: df_ml
# ============================================================

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ------------------------------------------------------------
# 1. TEXT, SENTIMENT FEATURES AND TARGET
# ------------------------------------------------------------

X_text = df_ml['lemmatized_tweet'].fillna('')

sentiment_features = [
    'negative',
    'positive',
    'neutral',
    'compound'
]

X_sentiment = df_ml[sentiment_features].fillna(0)
y = df_ml['cyberbullying_type']

# ------------------------------------------------------------
# 2. LABEL ENCODING
# ------------------------------------------------------------

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nClasses:")
print(label_encoder.classes_)

label_mapping = dict(enumerate(label_encoder.classes_))

print("\nLabel Mapping:")
for number, category in label_mapping.items():
    print(f"{number} -> {category}")

# ------------------------------------------------------------
# 3. TRAIN-TEST SPLIT
# ------------------------------------------------------------

(
    X_text_train,
    X_text_test,
    X_sent_train,
    X_sent_test,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_sentiment,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain-Test Split")
print("-------------------------")
print("Training samples:", len(X_text_train))
print("Testing samples :", len(X_text_test))

# ------------------------------------------------------------
# 4. WORD TF-IDF VECTORIZATION
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("\nWord TF-IDF Features")
print("-------------------------")
print("X_train TF-IDF:", X_train_tfidf.shape)
print("X_test TF-IDF :", X_test_tfidf.shape)

# ------------------------------------------------------------
# 5. COMBINE TF-IDF + SENTIMENT FEATURES
# ------------------------------------------------------------

X_sent_train_sparse = csr_matrix(
    X_sent_train.to_numpy(dtype=float)
)

X_sent_test_sparse = csr_matrix(
    X_sent_test.to_numpy(dtype=float)
)

X_train_combined = hstack(
    [X_train_tfidf, X_sent_train_sparse],
    format='csr'
)

X_test_combined = hstack(
    [X_test_tfidf, X_sent_test_sparse],
    format='csr'
)

print("\nCombined Features: Word TF-IDF + Sentiment")
print("-------------------------")
print("X_train:", X_train_combined.shape)
print("X_test :", X_test_combined.shape)



Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Label Mapping:
0 -> age
1 -> ethnicity
2 -> gender
3 -> not_cyberbullying
4 -> other_cyberbullying
5 -> religion

Train-Test Split
-------------------------
Training samples: 29821
Testing samples : 12781

Word TF-IDF Features
-------------------------
X_train TF-IDF: (29821, 10000)
X_test TF-IDF : (12781, 10000)

Combined Features: Word TF-IDF + Sentiment
-------------------------
X_train: (29821, 10004)
X_test : (12781, 10004)


In [4]:
# ============================================================
# hexa class + WORD TF-IDF + SENTIMENT ANALYSIS
# NO BALANCING + LOGISTIC REGRESSION
# FIXED HYPERPARAMETER: C = 1
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import numpy as np

# ------------------------------------------------------------
# 1. MODEL WITH FIXED HYPERPARAMETERS
# ------------------------------------------------------------

lr_none = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL, UNBALANCED DATA
# ------------------------------------------------------------

print("Training Word TF-IDF + Sentiment + No Balancing + LR...")
print("Fixed hyperparameters: C=1, max_iter=1000, random_state=42")

lr_none.fit(
    X_train_combined,
    y_train
)

# ------------------------------------------------------------
# 3. BEST MODEL
# ------------------------------------------------------------

best_none = lr_none

# ------------------------------------------------------------
# 4. TEST PREDICTION
# ------------------------------------------------------------

y_pred_none = best_none.predict(X_test_combined)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

none_accuracy = accuracy_score(y_test, y_pred_none)

none_weighted_f1 = f1_score(
    y_test, y_pred_none, average='weighted'
)

none_macro_f1 = f1_score(
    y_test, y_pred_none, average='macro'
)

print("\n==============================================")
print("WORD TF-IDF + SENTIMENT + NO BALANCING + LR")
print("==============================================")

print(f"Accuracy     : {none_accuracy:.4f}")
print(f"Weighted F1  : {none_weighted_f1:.4f}")
print(f"Macro F1     : {none_macro_f1:.4f}")

# ------------------------------------------------------------
# 6. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(classification_report(
    y_test,
    y_pred_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Word TF-IDF + Sentiment + No Balancing + LR...
Fixed hyperparameters: C=1, max_iter=1000, random_state=42

WORD TF-IDF + SENTIMENT + NO BALANCING + LR
Accuracy     : 0.8436
Weighted F1  : 0.8450
Macro F1     : 0.8248

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.97      0.96      2388
          ethnicity       0.97      0.98      0.98      2228
             gender       0.92      0.83      0.87      2243
  not_cyberbullying       0.59      0.61      0.60      1931
other_cyberbullying       0.57      0.60      0.59      1602
           religion       0.95      0.95      0.95      2389

           accuracy                           0.84     12781
          macro avg       0.83      0.82      0.82     12781
       weighted avg       0.85      0.84      0.85     12781



In [5]:
# ============================================================
# HEXA CLASS + WORD TF-IDF + SENTIMENT ANALYSIS
# NO BALANCING + LINEAR SVM
# FIXED HYPERPARAMETER: C = 1
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import numpy as np

# ------------------------------------------------------------
# 1. MODEL WITH FIXED HYPERPARAMETERS
# ------------------------------------------------------------

svm_none = LinearSVC(
    C=1,
    max_iter=10000,
    random_state=42
)

# ------------------------------------------------------------
# 2. TRAIN ON ORIGINAL, UNBALANCED DATA
# ------------------------------------------------------------

print("Training Word TF-IDF + Sentiment + No Balancing + Linear SVM...")
print("Fixed hyperparameters: C=1, max_iter=10000, random_state=42")

svm_none.fit(
    X_train_combined,
    y_train
)

# ------------------------------------------------------------
# 3. BEST MODEL
# ------------------------------------------------------------

best_svm_none = svm_none

# ------------------------------------------------------------
# 4. TEST PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(X_test_combined)

# ------------------------------------------------------------
# 5. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(y_test, y_pred_svm_none)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted',
    zero_division=0
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro',
    zero_division=0
)

print("\n==============================================")
print("WORD TF-IDF + SENTIMENT + NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 6. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(classification_report(
    y_test,
    y_pred_svm_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Word TF-IDF + Sentiment + No Balancing + Linear SVM...
Fixed hyperparameters: C=1, max_iter=10000, random_state=42

WORD TF-IDF + SENTIMENT + NO BALANCING + LINEAR SVM
Accuracy     : 0.8420
Weighted F1  : 0.8421
Macro F1     : 0.8215

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.95      0.97      0.96      2388
          ethnicity       0.97      0.98      0.98      2228
             gender       0.90      0.85      0.87      2243
  not_cyberbullying       0.59      0.58      0.58      1931
other_cyberbullying       0.57      0.59      0.58      1602
           religion       0.95      0.95      0.95      2389

           accuracy                           0.84     12781
          macro avg       0.82      0.82      0.82     12781
       weighted avg       0.84      0.84      0.84     12781



In [6]:
# ============================================================
# EXPERIMENT: CHARACTER TF-IDF + SENTIMENT FEATURES
# DATASET: df_ml
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ------------------------------------------------------------
# 1. CHARACTER TF-IDF VECTORIZATION
# ------------------------------------------------------------

char_tfidf = TfidfVectorizer(
    analyzer='char',
    max_features=10000,
    ngram_range=(3, 5)
)

X_train_char_tfidf = char_tfidf.fit_transform(X_text_train)
X_test_char_tfidf = char_tfidf.transform(X_text_test)

print("Character TF-IDF Features")
print("-------------------------")
print("X_train:", X_train_char_tfidf.shape)
print("X_test :", X_test_char_tfidf.shape)

# ------------------------------------------------------------
# 2. COMBINE CHARACTER TF-IDF + SENTIMENT
# ------------------------------------------------------------

X_train_char_combined = hstack(
    [X_train_char_tfidf, X_sent_train_sparse],
    format='csr'
)

X_test_char_combined = hstack(
    [X_test_char_tfidf, X_sent_test_sparse],
    format='csr'
)

print("\nCharacter TF-IDF + Sentiment")
print("-------------------------")
print("X_train:", X_train_char_combined.shape)
print("X_test :", X_test_char_combined.shape)

Character TF-IDF Features
-------------------------
X_train: (29821, 10000)
X_test : (12781, 10000)

Character TF-IDF + Sentiment
-------------------------
X_train: (29821, 10004)
X_test : (12781, 10004)


In [7]:
# ============================================================
# CHARACTER TF-IDF + SENTIMENT
# NO BALANCING + LOGISTIC REGRESSION
# C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

lr_char_none = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

print("Training Character TF-IDF + Sentiment + LR...")
print("C=1, max_iter=1000, random_state=42, no balancing")

lr_char_none.fit(X_train_char_combined, y_train)

y_pred_lr_char_none = lr_char_none.predict(X_test_char_combined)

print("\n==============================================")
print("CHARACTER TF-IDF + SENTIMENT + NO BALANCING + LR")
print("==============================================")
print(f"Accuracy     : {accuracy_score(y_test, y_pred_lr_char_none):.4f}")
print(f"Weighted F1  : {f1_score(y_test, y_pred_lr_char_none, average='weighted', zero_division=0):.4f}")
print(f"Macro F1     : {f1_score(y_test, y_pred_lr_char_none, average='macro', zero_division=0):.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred_lr_char_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Character TF-IDF + Sentiment + LR...
C=1, max_iter=1000, random_state=42, no balancing

CHARACTER TF-IDF + SENTIMENT + NO BALANCING + LR
Accuracy     : 0.8478
Weighted F1  : 0.8493
Macro F1     : 0.8284

Classification Report
                     precision    recall  f1-score   support

                age       0.96      0.98      0.97      2388
          ethnicity       0.98      0.98      0.98      2228
             gender       0.92      0.84      0.88      2243
  not_cyberbullying       0.60      0.62      0.61      1931
other_cyberbullying       0.56      0.59      0.58      1602
           religion       0.95      0.95      0.95      2389

           accuracy                           0.85     12781
          macro avg       0.83      0.83      0.83     12781
       weighted avg       0.85      0.85      0.85     12781



In [8]:
# ============================================================
# CHARACTER TF-IDF + SENTIMENT
# NO BALANCING + LINEAR SVM
# C = 1, max_iter = 1000
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

svm_char_none = LinearSVC(
    C=1,
    max_iter=1000,
    random_state=42
)

print("Training Character TF-IDF + Sentiment + Linear SVM...")
print("C=1, max_iter=1000, random_state=42, no balancing")

svm_char_none.fit(X_train_char_combined, y_train)

y_pred_svm_char_none = svm_char_none.predict(X_test_char_combined)

print("\n==============================================")
print("CHARACTER TF-IDF + SENTIMENT + NO BALANCING + SVM")
print("==============================================")
print(f"Accuracy     : {accuracy_score(y_test, y_pred_svm_char_none):.4f}")
print(f"Weighted F1  : {f1_score(y_test, y_pred_svm_char_none, average='weighted', zero_division=0):.4f}")
print(f"Macro F1     : {f1_score(y_test, y_pred_svm_char_none, average='macro', zero_division=0):.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred_svm_char_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Character TF-IDF + Sentiment + Linear SVM...
C=1, max_iter=1000, random_state=42, no balancing

CHARACTER TF-IDF + SENTIMENT + NO BALANCING + SVM
Accuracy     : 0.8459
Weighted F1  : 0.8459
Macro F1     : 0.8246

Classification Report
                     precision    recall  f1-score   support

                age       0.96      0.98      0.97      2388
          ethnicity       0.98      0.99      0.98      2228
             gender       0.90      0.85      0.88      2243
  not_cyberbullying       0.60      0.58      0.59      1931
other_cyberbullying       0.56      0.59      0.57      1602
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12781
          macro avg       0.82      0.82      0.82     12781
       weighted avg       0.85      0.85      0.85     12781



In [9]:
# ============================================================
# EXPERIMENT: COMBINED WORD + CHARACTER TF-IDF + SENTIMENT
# DATASET: df_ml
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# ------------------------------------------------------------
# 1. WORD TF-IDF
# ------------------------------------------------------------

word_tfidf_comb = TfidfVectorizer(
    analyzer='word',
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_word_comb = word_tfidf_comb.fit_transform(X_text_train)
X_test_word_comb = word_tfidf_comb.transform(X_text_test)

# ------------------------------------------------------------
# 2. CHARACTER TF-IDF
# ------------------------------------------------------------

char_tfidf_comb = TfidfVectorizer(
    analyzer='char',
    max_features=10000,
    ngram_range=(3, 5)
)

X_train_char_comb = char_tfidf_comb.fit_transform(X_text_train)
X_test_char_comb = char_tfidf_comb.transform(X_text_test)

# ------------------------------------------------------------
# 3. COMBINE WORD + CHARACTER FEATURES
# ------------------------------------------------------------

X_train_word_char = hstack(
    [X_train_word_comb, X_train_char_comb],
    format='csr'
)

X_test_word_char = hstack(
    [X_test_word_comb, X_test_char_comb],
    format='csr'
)

# ------------------------------------------------------------
# 4. APPEND SENTIMENT FEATURES
# ------------------------------------------------------------

X_train_word_char_sent = hstack(
    [X_train_word_char, X_sent_train_sparse],
    format='csr'
)

X_test_word_char_sent = hstack(
    [X_test_word_char, X_sent_test_sparse],
    format='csr'
)

print("Combined Word + Character TF-IDF + Sentiment")
print("---------------------------------------------")
print("Word train features      :", X_train_word_comb.shape)
print("Character train features :", X_train_char_comb.shape)
print("Combined train features  :", X_train_word_char_sent.shape)
print("Combined test features   :", X_test_word_char_sent.shape)

Combined Word + Character TF-IDF + Sentiment
---------------------------------------------
Word train features      : (29821, 10000)
Character train features : (29821, 10000)
Combined train features  : (29821, 20004)
Combined test features   : (12781, 20004)


In [10]:
# ============================================================
# WORD + CHARACTER TF-IDF + SENTIMENT
# NO BALANCING + LOGISTIC REGRESSION
# C = 1, max_iter = 1000
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

lr_word_char_none = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

print("Training Word + Character TF-IDF + Sentiment + LR...")
print("C=1, max_iter=1000, random_state=42, no balancing")

lr_word_char_none.fit(X_train_word_char_sent, y_train)

y_pred_lr_word_char_none = lr_word_char_none.predict(
    X_test_word_char_sent
)

print("\n================================================")
print("WORD + CHARACTER TF-IDF + SENTIMENT + NO BALANCING + LR")
print("================================================")
print(f"Accuracy     : {accuracy_score(y_test, y_pred_lr_word_char_none):.4f}")
print(f"Weighted F1  : {f1_score(y_test, y_pred_lr_word_char_none, average='weighted', zero_division=0):.4f}")
print(f"Macro F1     : {f1_score(y_test, y_pred_lr_word_char_none, average='macro', zero_division=0):.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred_lr_word_char_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Word + Character TF-IDF + Sentiment + LR...
C=1, max_iter=1000, random_state=42, no balancing

WORD + CHARACTER TF-IDF + SENTIMENT + NO BALANCING + LR
Accuracy     : 0.8515
Weighted F1  : 0.8529
Macro F1     : 0.8327

Classification Report
                     precision    recall  f1-score   support

                age       0.96      0.98      0.97      2388
          ethnicity       0.98      0.99      0.98      2228
             gender       0.92      0.84      0.88      2243
  not_cyberbullying       0.60      0.61      0.61      1931
other_cyberbullying       0.58      0.61      0.60      1602
           religion       0.96      0.96      0.96      2389

           accuracy                           0.85     12781
          macro avg       0.83      0.83      0.83     12781
       weighted avg       0.86      0.85      0.85     12781



In [11]:
# ============================================================
# WORD + CHARACTER TF-IDF + SENTIMENT
# NO BALANCING + LINEAR SVM
# C = 1, max_iter = 1000
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

svm_word_char_none = LinearSVC(
    C=1,
    max_iter=1000,
    random_state=42
)

print("Training Word + Character TF-IDF + Sentiment + Linear SVM...")
print("C=1, max_iter=1000, random_state=42, no balancing")

svm_word_char_none.fit(X_train_word_char_sent, y_train)

y_pred_svm_word_char_none = svm_word_char_none.predict(
    X_test_word_char_sent
)

print("\n================================================")
print("WORD + CHARACTER TF-IDF + SENTIMENT + NO BALANCING + SVM")
print("================================================")
print(f"Accuracy     : {accuracy_score(y_test, y_pred_svm_word_char_none):.4f}")
print(f"Weighted F1  : {f1_score(y_test, y_pred_svm_word_char_none, average='weighted', zero_division=0):.4f}")
print(f"Macro F1     : {f1_score(y_test, y_pred_svm_word_char_none, average='macro', zero_division=0):.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred_svm_word_char_none,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

Training Word + Character TF-IDF + Sentiment + Linear SVM...
C=1, max_iter=1000, random_state=42, no balancing

WORD + CHARACTER TF-IDF + SENTIMENT + NO BALANCING + SVM
Accuracy     : 0.8467
Weighted F1  : 0.8465
Macro F1     : 0.8254

Classification Report
                     precision    recall  f1-score   support

                age       0.96      0.98      0.97      2388
          ethnicity       0.98      0.99      0.98      2228
             gender       0.90      0.86      0.88      2243
  not_cyberbullying       0.59      0.58      0.59      1931
other_cyberbullying       0.57      0.58      0.58      1602
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12781
          macro avg       0.83      0.83      0.83     12781
       weighted avg       0.85      0.85      0.85     12781

